# Predict quantities

{meth}`~liesel.model.Model.predict` evaluates named quantities at supplied
parameter draws. It propagates their uncertainty through the calculations in
the graph. For new random responses, use {doc}`model-simulation`.

## Choose outputs

These examples use `model` and posterior `samples` from
{doc}`tutorials/notebooks/12-model-predictions`. Both the mean `mu` and
standard deviation `sigma` depend on `x` in that model.

In [1]:
import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

In [2]:
# Python cell fragment: imports are visible in each guide.
x_data = jnp.linspace(-1.0, 1.0, 120)
true_mean = 1.0 + 2.0 * x_data
true_scale = jnp.exp(-0.4 + 0.7 * x_data)
y_data = true_mean + true_scale * jax.random.normal(jax.random.key(8), (120,))

beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=2.5),
    name="beta",
)
gamma = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="gamma",
)
x = lsl.Var.new_obs(x_data, name="x")
mu = lsl.Var.new_calc(
    lambda x, b: b[0] + b[1] * x,
    x,
    beta,
    name="mu",
)
log_sigma = lsl.Var.new_calc(
    lambda x, g: g[0] + g[1] * x,
    x,
    gamma,
    name="log_sigma",
)
sigma = lsl.Var.new_calc(jnp.exp, log_sigma, name="sigma")
y = lsl.Var.new_obs(
    y_data,
    dist=lsl.Dist(tfd.Normal, loc=mu, scale=sigma),
    name="y",
)
model = lsl.Model(y)

for parameter in model.parameters.values():
    parameter.inference = gs.MCMCSpec(
        gs.NUTSKernel,
        kernel_group="regression",
        jitter_dist=tfd.Normal(loc=0.0, scale=0.1),
    )
results = gs.LieselMCMC(model).run_for_epochs(
    seed=31,
    num_chains=4,
    adaptation=500,
    posterior=500,
    show_progress=False,
)
samples = results.get_posterior_samples()


liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


In [3]:
x_grid = jnp.linspace(-1.0, 1.0, 60)
predicted = model.predict(
    samples,
    predict=["mu", "sigma"],
    newdata={"x": x_grid},
)

In [4]:
{name: value.shape for name, value in predicted.items()}

{'mu': (4, 500, 60), 'sigma': (4, 500, 60)}

Each array has shape `(4, 500, 60)` for the tutorial's four chains and
500 retained draws. Its last axis indexes grid points. The `predict` entries
are variable or node names, not distribution argument names such as `loc`.
Values saved for weak variables in `samples` are ignored and recalculated.

## Supply new data

`newdata` replaces named inputs for the evaluation. Omit it to use the current
data; omitted inputs keep their current values. Supply compatible arrays for
all covariates that change together. If the mean and scale use separate design
inputs `X` and `Z`, provide both with aligned rows; a one-dimensional slice must
state which other covariate is fixed. See
{ref}`different-covariates` in the worked tutorial. Keys must not overlap between `samples`
and `newdata`. Include draws for every parameter whose posterior uncertainty
you intend to propagate: missing strong inputs keep their current values.

Selecting only `mu` and `sigma` allows prediction to use their parental
submodel, so the old response length does not matter. Predicting response
log densities also requires response values matching the new covariates.

With default outputs or model-total nodes, the current implementation uses the
full model and applies the new-data state to it. Use `model.copy().predict(...)`
when you need to preserve its state on those paths. Explicit output selections
as above operate on a copied parental submodel.

## Summarize predictions

In [5]:
summary = gs.SamplesSummary(
    predicted,
    quantiles=(0.05, 0.95),
    which=["quantiles"],
)

mean_interval = summary.quantities["quantile"]["mu"]

In [6]:
mean_interval.shape

(2, 60)

The shape is `(2, 60)`: lower and upper limits for each grid point. These are
pointwise 90% credible intervals for the conditional mean. They describe neither
new response variation nor simultaneous coverage of the entire curve.
Summarize `predicted["sigma"]` similarly to inspect uncertainty in the spread.

See {doc}`model-distributions` for pointwise log likelihoods. For large prediction
jobs, the `chunk_size` option bounds parallel intermediate calculations; it
does not reduce the memory needed for the returned arrays.